# 03. Ablation, McNemar 검정과 비용 분석

목표: 논문 표 1~2의 공개 수치로 구성 요소별 증가분, paired exact McNemar p-value와 비용 대비 성능을 계산합니다. 이 notebook은 보고된 집계치를 확인하며 원 실험을 재실행하지 않습니다.

In [ ]:
from math import comb

results = {
    'Opus 4.6': {'B0': 32.3, 'B1': 37.9, 'L1': 39.5, 'L2': 51.6, 'L3': 62.1},
    'DeepSeek V4 Pro': {'B0': 29.0, 'B1': 31.4, 'L1': 31.4, 'L2': 39.5, 'L3': 50.8},
}
costs = {
    'Opus 4.6': {'B0': 2.96, 'B1': 17.76, 'L1': 5.38, 'L2': 15.59, 'L3': 19.45},
    'DeepSeek V4 Pro': {'B0': 0.42, 'B1': 2.52, 'L1': 0.77, 'L2': 1.93, 'L3': 2.46},
}

for model, row in results.items():
    print(model, {
        'division': round(row['L1'] - row['B0'], 1),
        'negotiation': round(row['L2'] - row['L1'], 1),
        'passive': round(row['L3'] - row['L2'], 1),
    })

## Exact McNemar 검정

두 시스템 결과가 다른 discordant pair만 사용합니다. 귀무가설에서 어느 시스템이 이길 확률은 0.5이며, 논문처럼 양측 exact binomial p-value를 계산합니다.

In [ ]:
def exact_mcnemar(wins: int, losses: int) -> float:
    n = wins + losses
    tail = sum(comb(n, k) for k in range(min(wins, losses) + 1)) / (2 ** n)
    return min(1.0, 2 * tail)

for model, pair in {'Opus 4.6': (15, 2), 'DeepSeek V4 Pro': (17, 3)}.items():
    p = exact_mcnemar(*pair)
    print(model, pair, f'p={p:.4f}')

assert round(exact_mcnemar(15, 2), 4) == 0.0023
assert round(exact_mcnemar(17, 3), 4) == 0.0026

In [ ]:
for model in results:
    r, c = results[model], costs[model]
    print(f'\n{model}')
    print('L3 vs B1 accuracy gain:', round(r['L3'] - r['B1'], 1), '%p')
    print('L3 vs B1 cost difference:', round(c['L3'] - c['B1'], 2), 'USD/task')
    print('passive marginal gain per extra dollar:', round((r['L3'] - r['L2']) / (c['L3'] - c['L2']), 2), '%p/USD')

## 올바른 해석

p-value는 효과 크기나 다른 benchmark로의 일반화를 보장하지 않습니다. 비용 비교도 당시 API 가격에 의존합니다. 후속 실험에서는 task별 paired outcome, confidence interval, token 수, wall-clock latency와 메시지로 인해 퇴행한 rubric을 함께 보고해야 합니다.